# Chapter 6 — Ensembles of stars: Liouville's theorem and phase mixing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/06_phase_mixing.ipynb)

**Goal.** So far we followed one star at a time. A galaxy contains $10^{11}$ stars, and we describe them
by their density in phase space. This chapter shows how that density evolves, why a galaxy that is
disturbed settles back into a steady state without any collisions between stars, and how the Gaia
satellite caught the Milky Way's disk in the act of settling.

**You will learn**
1. the **phase-space density** $f(\mathbf x, \mathbf v, t)$ of an ensemble of stars;
2. **Liouville's theorem** and the **collisionless Boltzmann equation**: $f$ is constant along orbits,
   so phase-space area is conserved;
3. **phase mixing**: stars with different orbital frequencies drift apart in angle, and the
   *coarse-grained* density becomes steady (the **Jeans theorem**);
4. the **phase spiral** of the Milky Way disk, and how to read the time since a disturbance from it.

**Units.** $M_\odot$, kpc, km/s, as in Chapters 4–5. Times are in AGAMA's unit 1 kpc/(km/s) = 977.8 Myr;
frequencies in km/s/kpc.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object
TIME_UNIT_MYR = agama.getUnits()['time']           # 977.8 Myr

import os
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)                    # fixed seed: the notebook gives the same numbers every run

## 6.1 An ensemble in phase space, and a case with no mixing

Describe $N$ stars by the **phase-space density** $f(\mathbf x, \mathbf v, t)$: the number of stars per unit
volume of the six-dimensional space of positions and velocities, so that
$\int f\, d^3x\, d^3v = N$. With $N = 10^{11}$ we never follow every star; instead we sample $f$ with
a manageable number of test particles and move each one along its orbit.

To draw the ensemble on paper we restrict it to two dimensions. All our stars move in the $(x, y)$
plane of a spherical potential with the **same angular momentum** $L$. Each star's motion is then fixed
by its radius $r$ and radial velocity $v_r$ (Chapter 1: the radial motion is that of a particle in the
effective potential $\Phi_{\rm eff}(r) = \Phi(r) + L^2/2r^2$), and the ensemble is a cloud of points in
the $(r, v_r)$ plane. We start with a uniform rectangular **patch** of points.

The simplest case is the **harmonic potential** $\Phi = \tfrac12 \Omega^2 r^2$ — the potential inside a
uniform-density sphere. AGAMA has it as `type='Harmonic'`. Every orbit in it is an ellipse centred on
the origin, and **every** orbit has the same radial period $T_r = \pi/\Omega$, whatever its energy.
The prediction: the patch moves and deforms, but after exactly one radial period every star is back
where it started. This is our negative control: no mixing when all frequencies are equal.

In [ ]:
def make_patch(n, r_c, dr, vr_c, dvr, L):
    """n stars uniformly in the rectangle r_c +- dr/2, vr_c +- dvr/2, all with angular momentum L.
    Returns AGAMA initial conditions (x, y, z, vx, vy, vz): stars on the x-axis, v_y = L/r."""
    r = r_c + dr * (rng.random(n) - 0.5)
    vr = vr_c + dvr * (rng.random(n) - 0.5)
    zero = np.zeros(n)
    return np.column_stack([r, zero, zero, vr, L / r, zero])

def r_vr(xv):
    """Radius and radial velocity v_r = (x v_x + y v_y)/r of points xv[..., 0:6] in the plane z = 0.
    The `...` lets it work on arrays of any shape whose last axis has the 6 coordinates."""
    r = np.hypot(xv[..., 0], xv[..., 1])
    return r, (xv[..., 0] * xv[..., 3] + xv[..., 1] * xv[..., 4]) / r

def run(pot, ic, time, nsnap):
    """Integrate all orbits; return the times and an array of shape (N stars, nsnap, 6).
    agama.orbit with N initial conditions returns an (N, 2) object array: column 0 holds each star's
    times, column 1 its trajectory; np.stack turns the list of trajectories into one 3-d array."""
    orbits = agama.orbit(potential=pot, ic=ic, time=time, trajsize=nsnap, dtype=float, verbose=False)
    return orbits[0, 0], np.stack(orbits[:, 1])

Omega = 50.0                                         # km/s/kpc
harm = agama.Potential(type='Harmonic', Omega=Omega) # Phi = Omega^2 r^2 / 2
L = 500.0                                            # kpc km/s, the same for all stars
ic = make_patch(4000, r_c=8.0, dr=1.0, vr_c=0.0, dvr=40.0, L=L)

T_r = np.pi / Omega                                  # radial period, the same for every orbit
t, X = run(harm, ic, time=T_r, nsnap=9)              # 9 snapshots: t = 0, T/8, ..., T
r, vr = r_vr(X)                                      # each of shape (N stars, 9 snapshots)

fig, ax = plt.subplots(figsize=(7, 5))
for k in range(9):
    ax.plot(r[:, k], vr[:, k], ',', color=plt.cm.viridis(k / 8))
    if k < 8:                                        # k = 8 lands on top of k = 0
        ax.text(r[:, k].mean(), vr[:, k].mean(), '0/8 = 8/8' if k == 0 else f'{k}/8', ha='center', fontsize=9)
ax.set_xlabel('$r$ [kpc]'); ax.set_ylabel('$v_r$ [km/s]')
ax.set_title('harmonic potential: the patch at $t = k\\,T_r/8$')
plt.show()

dmax = np.max(np.abs(r[:, -1] - r[:, 0])), np.max(np.abs(vr[:, -1] - vr[:, 0]))
print(f'radial period {T_r * TIME_UNIT_MYR:.1f} Myr; after one period: max |dr| = {dmax[0]:.1e} kpc, '
      f'max |dv_r| = {dmax[1]:.1e} km/s')
assert dmax[0] < 1e-6 and dmax[1] < 1e-4

**Figure 6.1.** A patch of 4000 stars in the $(r, v_r)$ plane of the harmonic potential at $t = k\,T_r/8$, $k = 0, \dots, 8$ (colour from dark to light); the labels sit at the patch centre at each time, and $k = 8$ lands on $k = 0$.

In Figure 6.1 the patch goes round the point of the circular orbit, stretching and squeezing, and returns intact
(to the integrator's accuracy). Watch its area: it changes shape but never looks bigger or smaller.
That is the subject of the next section.

## 6.2 Liouville's theorem and the collisionless Boltzmann equation

Stars in a galaxy almost never collide or deflect each other individually (Binney & Tremaine §1.2: the
relaxation time of a galaxy is far longer than its age). Each star moves in the smooth potential,
and the number of stars is conserved. Two facts follow.

**The flow in phase space is incompressible.** Hamilton's equations $\dot x = v$, $\dot v = -\partial\Phi/\partial x$
move a phase-space point with the velocity $(\dot x, \dot v)$, whose divergence is
$\partial \dot x/\partial x + \partial \dot v/\partial v = 0 + 0$, because $\dot x = v$ does not depend on $x$ and
$\dot v$ does not depend on $v$. A patch of phase space can change shape but **not its volume**
(in our 2-d slice: its area). This is **Liouville's theorem**.

**$f$ is constant along every orbit.** The stars in a patch stay in it and its volume is fixed, so their
density stays the same: $df/dt = 0$ following a star. Written out, this is the
**collisionless Boltzmann equation** (CBE)

$$ \frac{\partial f}{\partial t} + \mathbf v\cdot\frac{\partial f}{\partial \mathbf x}
   - \frac{\partial \Phi}{\partial \mathbf x}\cdot\frac{\partial f}{\partial \mathbf v} = 0 . $$

We test Liouville's theorem in a potential where the radial period **does** depend on energy: the
isochrone (Chapter 3), with $M = 10^{11}\,M_\odot$ and $b = 3$ kpc. We follow the **boundary** of the
patch — 4000 points around the rectangle — and compute the area enclosed by this polygon with the
shoelace formula, $A = \tfrac12\left|\sum_i (r_i v_{r,i+1} - r_{i+1} v_{r,i})\right|$.

In [ ]:
iso = agama.Potential(type='Isochrone', mass=1e11, scaleRadius=3.0)

def patch_boundary(n, r_c, dr, vr_c, dvr, L):
    """n points going once around the edge of the rectangle used by make_patch, in order."""
    s = np.linspace(0, 4, n, endpoint=False)          # position along the perimeter: 0-1, 1-2, 2-3, 3-4 = four sides
    u = np.clip(np.where(s < 2, s, 3 - s), 0, 1) - 0.5    # 0 -> 1 on side 1, stays 1, 1 -> 0 on side 3, stays 0
    w = np.clip(np.where(s < 3, s - 1, 4 - s), 0, 1) - 0.5
    r, vr = r_c + dr * u, vr_c + dvr * w
    zero = np.zeros(n)
    return np.column_stack([r, zero, zero, vr, L / r, zero])

def shoelace(x, y):
    """Area of the polygon with vertices (x[i], y[i]) in order. np.roll(x, -1) is x shifted by one: x[i+1]."""
    return 0.5 * abs(np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y))

edge = patch_boundary(4000, r_c=10.0, dr=1.0, vr_c=0.0, dvr=40.0, L=L)
ic = make_patch(4000, r_c=10.0, dr=1.0, vr_c=0.0, dvr=40.0, L=L)

# AGAMA's ActionFinder gives each orbit's radial frequency Omega_r = 2 pi / T_r (Chapter 3)
af = agama.ActionFinder(iso)
T_iso = 2 * np.pi / np.median(af(ic, frequencies=True)[1][:, 0])
t, Xe = run(iso, edge, time=4 * T_iso, nsnap=5)     # snapshots at 0, 1, 2, 3, 4 radial periods
t, Xi = run(iso, ic, time=4 * T_iso, nsnap=5)
re, vre = r_vr(Xe); ri, vri = r_vr(Xi)

fig, ax = plt.subplots(1, 5, figsize=(17, 3.6), sharex=True, sharey=True)
area = []
for k in range(5):
    area.append(shoelace(re[:, k], vre[:, k]))
    ax[k].plot(ri[:, k], vri[:, k], ',', color='C0')
    ax[k].plot(re[:, k], vre[:, k], '-', color='k', lw=0.6)
    ax[k].set_title(f'$t = {k}\\,T_r$:  area {area[k]:.4f}', fontsize=10)
    ax[k].set_xlabel('$r$ [kpc]')
ax[0].set_ylabel('$v_r$ [km/s]')
plt.tight_layout(); plt.show()

area = np.array(area)
print(f'radial period {T_iso * TIME_UNIT_MYR:.0f} Myr; area/initial area - 1 = {np.round(area / area[0] - 1, 9)}')
assert np.all(np.abs(area / area[0] - 1) < 1e-5)

**Figure 6.2.** A patch in the isochrone potential at $t = 0, 1, 2, 3, 4$ radial periods: the stars (blue) and the boundary of the patch (black), with the enclosed area in each panel's title.

In Figure 6.2 the patch is sheared into a long, thin arc, but its area (in kpc km/s) stays the same to better than
one part in $10^6$. The shearing happens because stars at the outer edge of the patch have more energy
and, in the isochrone, a longer radial period: they fall behind. Keep going and the arc wraps around
again and again. That is phase mixing.

## 6.3 Phase mixing

**Angles and actions.** Chapter 3 introduced the radial action $J_r$ and frequency $\Omega_r$. Each orbit
also has a radial **angle** $\theta_r$, which runs from $0$ (pericentre) to $\pi$ (apocentre) and back to
$2\pi$, growing at a constant rate:

$$ J_r = {\rm const}, \qquad \theta_r(t) = \theta_r(0) + \Omega_r(J_r)\, t . $$

In angle–action variables the whole problem is solved, and phase mixing is easy to see: stars with
different $J_r$ have different $\Omega_r$, so their angles drift apart linearly in time. `ActionFinder`
returns the angles if asked: `af(xv, angles=True)` returns three arrays — actions, angles and
frequencies — each with columns $(r, z, \phi)$.

We now follow a bigger patch for 50 radial periods (about 5 Gyr).

In [ ]:
N = 20000
ic = make_patch(N, r_c=10.0, dr=1.0, vr_c=0.0, dvr=40.0, L=L)
J0, th0, Om0 = af(ic, angles=True)                   # actions, angles, frequencies at t = 0
Jr, th_r0, Om_r = J0[:, 0], th0[:, 0], Om0[:, 0]     # column 0 = radial
print(f'Omega_r ranges over {np.ptp(Om_r) / np.median(Om_r):.1%} of its median')

t, X = run(iso, ic, time=50 * T_iso, nsnap=251)      # snapshots every 0.2 T_r
r, vr = r_vr(X)
times_T = t / T_iso                                  # time in units of the radial period

# Is theta_r(t) = theta_r(0) + Omega_r t?  Compare with ActionFinder's angle at the last snapshot.
_, th_end, _ = af(X[:, -1], angles=True)
dth = np.angle(np.exp(1j * (th_end[:, 0] - th_r0 - Om_r * t[-1])))   # difference wrapped into (-pi, pi]
print(f'max |theta_r(t) - theta_r(0) - Omega_r t| after 50 periods: {np.max(np.abs(dth)):.1e} rad')
assert np.max(np.abs(dth)) < 1e-4

fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for j, k in enumerate([0, 10, 50, 250]):            # t = 0, 2, 10, 50 T_r
    th_k = (th_r0 + Om_r * t[k]) % (2 * np.pi)       # angles from the linear law, wrapped into [0, 2 pi)
    ax[0, j].plot(r[:, k], vr[:, k], ',', alpha=0.5)
    ax[0, j].set_title(f'$t = {times_T[k]:.0f}\\,T_r$ ({t[k] * TIME_UNIT_MYR / 1e3:.1f} Gyr)')
    ax[0, j].set_xlabel('$r$ [kpc]')
    ax[1, j].plot(th_k, Jr, ',', alpha=0.5)
    ax[1, j].set_xlabel('$\\theta_r$ [rad]'); ax[1, j].set_xlim(0, 2 * np.pi)
ax[0, 0].set_ylabel('$v_r$ [km/s]'); ax[1, 0].set_ylabel('$J_r$ [kpc km/s]')
plt.tight_layout(); plt.show()

**Figure 6.3.** 20 000 stars of a patch in the isochrone potential at $t = 0, 2, 10, 50\,T_r$. Top: in $(r, v_r)$. Bottom: in angle–action variables $(\theta_r, J_r)$, with angles from the linear law $\theta_r(0) + \Omega_r t$.

Figure 6.3, top row: the patch in $(r, v_r)$ winds into a tightening spiral and ends up filling a ring. Bottom row:
the same stars in $(\theta_r, J_r)$. Each star moves horizontally (its $J_r$ is fixed) at speed $\Omega_r(J_r)$,
so the initial strip tilts, wraps around, and fills the band of actions uniformly in angle.

**Fine-grained and coarse-grained density.** The true ("fine-grained") $f$ of a star never changes
(Liouville), and the ensemble never really becomes smooth: it is an ever thinner, ever longer filament.
But any measurement — a histogram, a telescope's error bars — averages $f$ over a finite cell. This
**coarse-grained** density does change: the filament is spread over more cells, so the peak
coarse-grained density falls and the distribution becomes smooth and steady.

Two measures of mixing:
1. the peak of the histogram of the ensemble in $(r, v_r)$, in units of the initial density;
2. the angle "clumpiness" $|\langle e^{i\theta_r}\rangle|$, which is 1 if all stars have the same angle and
   0 if angles are uniform. The linear law predicts it at any time from the $t = 0$ angles and
   frequencies alone: $|\langle e^{i(\theta_r(0) + \Omega_r t)}\rangle|$.

In [ ]:
lims = [[r.min(), r.max()], [vr.min(), vr.max()]]
cell = np.ptp(lims[0]) / 30 * np.ptp(lims[1]) / 30          # area of one histogram cell
f_initial = N / (1.0 * 40.0)                                 # stars per unit area in the starting rectangle
snaps = np.arange(0, 251, 5)                                 # every radial period
f_peak = [np.histogram2d(r[:, k], vr[:, k], bins=30, range=lims)[0].max() / cell / f_initial for k in snaps]

clump_pred = np.abs(np.mean(np.exp(1j * (th_r0[:, None] + Om_r[:, None] * t[None, :])), axis=0))  # (N, nt) -> nt
check = np.arange(0, 251, 18)                                # every 3.6 T_r (not a multiple of the ~10 T_r wiggle)
clump_meas = [np.abs(np.mean(np.exp(1j * af(X[:, k], angles=True)[1][:, 0]))) for k in check]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(times_T[snaps], f_peak)
ax[0].axhline(1, color='gray', ls=':')
ax[0].set_xlabel('$t / T_r$'); ax[0].set_ylabel('peak coarse-grained $f$ / initial $f$')
ax[1].semilogy(times_T, clump_pred, label='prediction from $t = 0$')
ax[1].semilogy(times_T[check], clump_meas, 'o', label='ActionFinder on the orbits')
ax[1].axhline(1 / np.sqrt(N), color='gray', ls=':', label='noise level $1/\\sqrt{N}$')
ax[1].set_xlabel('$t / T_r$'); ax[1].set_ylabel('$|\\langle e^{i\\theta_r}\\rangle|$'); ax[1].legend()
plt.tight_layout(); plt.show()

print(f'peak coarse-grained f / initial: {f_peak[0]:.2f} at t = 0, {f_peak[-1]:.2f} at 50 T_r')
print(f'max |prediction - measurement| of the clumpiness: {np.max(np.abs(clump_meas - clump_pred[check])):.1e}')
for t1, t2 in [(0, 10), (10, 20), (20, 30), (40, 50)]:
    w = (times_T >= t1) & (times_T <= t2)
    print(f'largest clumpiness between {t1:2d} and {t2:2d} T_r: {clump_pred[w].max():.3f}')
assert np.allclose(clump_meas, clump_pred[check], atol=1e-4)
assert f_peak[-1] < 0.3 * f_peak[0]

**Figure 6.4.** Left: the peak coarse-grained phase-space density in $(r, v_r)$ (30 × 30 cells), relative to the initial density, against time in radial periods. Right: the angle clumpiness $|\langle e^{i\theta_r}\rangle|$ predicted from the $t = 0$ angles and frequencies (line) and measured with `ActionFinder` (points); dotted: the noise level $1/\sqrt N$.

The points and the curve in Figure 6.4 (right) agree: mixing is nothing but the linear drift of angles.

The clumpiness does not fall smoothly. It drops to nearly zero after about 10 radial periods, rises
again, and keeps oscillating with an envelope that falls roughly as $1/t$. A simple model explains this:
if the frequencies were spread uniformly over a range $\Delta\Omega_r$, then
$|\langle e^{i\Omega_r t}\rangle| = |\sin(\Delta\Omega_r t/2)| \,/\, (\Delta\Omega_r t/2)$. It is zero whenever the
angles span exactly a multiple of $2\pi$, $\Delta\Omega_r\, t = 2\pi n$, i.e. every
$T_r\,\Omega_r/\Delta\Omega_r \approx 9$ periods here (our frequencies are not exactly uniform, hence ~10).
In between, the angles cover $2\pi$ plus a bit, and the leftover bit is a clump that shrinks as $1/t$.
Phase mixing is therefore slow: after 50 periods (8.7 Gyr) the angles still carry a few per cent of
structure, and the peak coarse-grained density has dropped to about 15% of its initial value.

**The Jeans theorem.** In the mixed state, the density depends on $J_r$ but not on $\theta_r$. Any $f$ that
depends on the phase-space coordinates only through integrals of motion — $f(J)$, or $f(E, L)$ — is
unchanged by orbital motion and solves the CBE with $\partial f/\partial t = 0$. That is the
**Jeans theorem**: steady states are functions of the integrals of motion. Phase mixing drives an
ensemble towards one.

Check: compare the histogram of radii at two times. At the start it changes completely in 0.2 radial
periods; at the end it is the same after 1 radial period, apart from Poisson noise. The statistic
$\chi^2 = \sum (n_1 - n_2)^2/(n_1 + n_2)$ over $K$ bins should be about $K$ if the two histograms differ
only by noise.

In [ ]:
def chi2_per_bin(a, b, bins=40):
    n1 = np.histogram(a, bins=bins, range=lims[0])[0]
    n2 = np.histogram(b, bins=bins, range=lims[0])[0]
    keep = n1 + n2 > 0
    return np.sum((n1 - n2)[keep]**2 / (n1 + n2)[keep]) / bins

early, late = chi2_per_bin(r[:, 0], r[:, 1]), chi2_per_bin(r[:, -6], r[:, -1])
print(f'chi^2 per bin: start (0 vs 0.2 T_r) = {early:.0f},  end (49 vs 50 T_r) = {late:.2f}')
assert early > 100 and late < 2

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(r[:, 0], bins=40, range=lims[0], histtype='step', label='$t = 0$')
ax.hist(r[:, 1], bins=40, range=lims[0], histtype='step', label='$t = 0.2\\,T_r$')
ax.hist(r[:, -6], bins=40, range=lims[0], histtype='step', label='$t = 49\\,T_r$')
ax.hist(r[:, -1], bins=40, range=lims[0], histtype='step', ls='--', label='$t = 50\\,T_r$')
ax.set_xlabel('$r$ [kpc]'); ax.set_ylabel('stars per bin'); ax.legend()
plt.show()

**Figure 6.5.** Histograms of the radii of the patch stars at $t = 0$ and $0.2\,T_r$, and at $49$ and $50\,T_r$ (dashed).

## 6.4 The phase spiral of the Milky Way disk

In 2018 the second Gaia data release revealed a spiral in the distribution of nearby disk stars in the
plane of height $z$ and vertical velocity $v_z$ (Antoja et al. 2018). It is phase mixing caught half-way:
something — perhaps a passage of the Sagittarius dwarf galaxy — disturbed the vertical motions of
the disk, and the disturbance has been winding up ever since because stars with larger vertical
oscillations have lower vertical frequencies (Chapter 5 §5.3: $\nu$ is largest for small oscillations).

We reproduce this with the Milky Way model of Chapters 4–5 (MWPotential2014):

1. **An equilibrium.** Put $10^5$ stars on circular orbits at $R = 8$ kpc, with Gaussian $z$
   ($\sigma = 0.3$ kpc) and $v_z$ ($\sigma = 20$ km/s). This is not yet steady, so we let each star run
   for a random time between 2 and 4 Gyr: phase mixing makes the vertical angles uniform, which by the
   Jeans theorem is a steady state. `agama.orbit` accepts one integration time per star.
2. **A kick.** Add $15$ km/s to every star's $v_z$, as an impulsive tug from a passing satellite would.
3. **Wind-up.** Integrate for 500 Myr and look at $(z, v_z)$.

In [ ]:
mw = agama.Potential(os.path.join(os.path.dirname(agama.__file__), 'data', 'MWPotential2014.ini'))
af_mw = agama.ActionFinder(mw)                       # axisymmetric: uses the Staeckel fudge (Chapter 9)

Ns, R0 = 100000, 8.0
vc0 = np.sqrt(-R0 * mw.force([R0, 0, 0])[0])         # circular speed at R0 (Chapter 0 §0.4)
z0, vz0 = rng.normal(0, 0.3, Ns), rng.normal(0, 20.0, Ns)
ic = np.column_stack([np.full(Ns, R0), np.zeros(Ns), z0, np.zeros(Ns), np.full(Ns, vc0), vz0])
orbits = agama.orbit(potential=mw, ic=ic, time=rng.uniform(2, 4, Ns), trajsize=2, dtype=float, verbose=False)
equil = np.array([traj[-1] for traj in orbits[:, 1]])   # each star's final point: the mixed equilibrium

kicked = equil.copy()
kicked[:, 5] += 15.0                                  # v_z -> v_z + 15 km/s for every star
t_end = 500 / TIME_UNIT_MYR                           # 500 Myr in AGAMA time units
t, Xk = run(mw, kicked, time=t_end, nsnap=6)          # snapshots every 100 Myr
t, Xq = run(mw, equil, time=t_end, nsnap=6)           # control: the same stars without the kick

fig, ax = plt.subplots(2, 6, figsize=(20, 7), sharex=True, sharey=True)
for k in range(6):
    for row, Y in enumerate([Xq, Xk]):
        ax[row, k].hist2d(Y[:, k, 2], Y[:, k, 5], bins=120, range=[[-1.2, 1.2], [-70, 70]], cmap='Greys')
    ax[0, k].set_title(f'{t[k] * TIME_UNIT_MYR:.0f} Myr')
    ax[1, k].set_xlabel('$z$ [kpc]')
ax[0, 0].set_ylabel('no kick\n$v_z$ [km/s]'); ax[1, 0].set_ylabel('kicked\n$v_z$ [km/s]')
plt.tight_layout(); plt.show()

zq = [np.histogram(Xq[:, k, 2], bins=40, range=(-1.2, 1.2))[0] for k in (0, -1)]
chi2_q = np.sum((zq[0] - zq[1])**2 / np.maximum(zq[0] + zq[1], 1)) / 40
print(f'control (no kick): chi^2 per bin between the z-histograms at 0 and 500 Myr = {chi2_q:.2f}')
assert chi2_q < 2

**Figure 6.6.** $10^5$ disk stars at $R = 8$ kpc in MWPotential2014 in the $(z, v_z)$ plane, every 100 Myr. Top: the phase-mixed equilibrium without a kick. Bottom: the same stars after a 15 km/s kick in $v_z$ at $t = 0$.

Figure 6.6, top row: the unperturbed population stays the same, as an equilibrium should. Bottom row: the kicked
population first moves as a whole, then shears into a spiral that winds tighter with time. The
spiral is the same physics as §6.3, now in the vertical direction: the angle $\theta_z$ grows at the
vertical frequency $\Omega_z(J_z)$, which falls with $J_z$, so the outer parts lag behind.

**The spiral is a clock.** Right after the kick, the extra stars sit at roughly the same vertical angle
$\theta_{z,0}$ for every $J_z$. At time $t$ the excess is at $\theta_{z,0} + \Omega_z(J_z)\,t$. So, in a single
snapshot, measure for stars in narrow bins of $J_z$ the angle where the excess lies (the phase of
$\langle e^{i\theta_z}\rangle$) and plot it against $\Omega_z$: the slope is the time since the kick. This uses
only today's positions and velocities, plus a model of the potential — exactly what an observer has.

In [ ]:
J, th, Om = af_mw(Xk[:, -1], angles=True)            # actions, angles, frequencies 500 Myr after the kick
Jz, th_z, Om_z = J[:, 1], th[:, 1], Om[:, 1]         # column 1 = vertical
edges = np.quantile(Jz, np.linspace(0.05, 0.9, 15))  # 14 bins with equal numbers of stars
phase, om = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    s = (Jz > lo) & (Jz < hi)
    phase.append(np.angle(np.mean(np.exp(1j * th_z[s]))))   # angle of the excess
    om.append(np.median(Om_z[s]))
phase, om = np.unwrap(phase), np.array(om)           # unwrap removes 2 pi jumps between neighbouring bins
slope, intercept = np.polyfit(om, phase, 1)          # straight-line fit: phase = slope * Omega_z + intercept

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(om, phase, 'o', label='excess angle in $J_z$ bins')
ax.plot(om, slope * om + intercept, '-', label=f'fit: slope = {slope * TIME_UNIT_MYR:.0f} Myr')
ax.set_xlabel('$\\Omega_z$ [km/s/kpc]'); ax.set_ylabel('phase of $\\langle e^{i\\theta_z}\\rangle$ [rad]'); ax.legend()
plt.show()

print(f'time since the kick from the spiral: {slope * TIME_UNIT_MYR:.1f} Myr (true: 500 Myr)')
print(f'Omega_z ranges from {om.min():.0f} to {om.max():.0f} km/s/kpc across the bins')
assert abs(slope / t_end - 1) < 0.02

**Figure 6.7.** Phase of $\langle e^{i\theta_z}\rangle$ of the kicked stars 500 Myr after the kick, in 14 bins of $J_z$, against the median vertical frequency $\Omega_z$ of each bin (points), with a straight-line fit whose slope is the time since the kick.

In Figure 6.7 the phase falls on a straight line and its slope recovers the 500 Myr since the kick. The same
reasoning applied to the Gaia spiral, with a Milky Way potential, gave a perturbation 300–900 Myr ago
(Antoja et al. 2018); the range reflects uncertainties in the potential, i.e. in $\Omega_z(J_z)$.

The real spiral is much fainter than ours: a few per cent in density, best seen when stars are
coloured by their mean $v_R$ or $v_\phi$. And our kick acted on a single radius; in the Galaxy, stars from
different radii, with different frequencies, pass through the solar neighbourhood. These complications
are the subject of current research.

## Summary

- An ensemble of stars is described by its phase-space density $f(\mathbf x, \mathbf v, t)$.
- Without collisions, $f$ is constant along orbits (collisionless Boltzmann equation) and phase-space
  volume is conserved (Liouville's theorem).
- When orbital frequencies depend on the orbit, angles drift apart: $\theta(t) = \theta(0) + \Omega(J)\,t$.
  The fine-grained $f$ becomes a thin filament; the coarse-grained $f$ becomes smooth and steady.
- Steady states are functions of the integrals of motion (Jeans theorem); in the harmonic potential,
  where all frequencies are equal, there is no mixing.
- The Gaia phase spiral is phase mixing in progress; its winding measures the time since the disk was
  disturbed.

| task | AGAMA |
|---|---|
| harmonic potential $\tfrac12\Omega^2 r^2$ | `agama.Potential(type='Harmonic', Omega=...)` |
| many orbits, one time per star | `agama.orbit(potential=, ic=(N, 6) array, time=(N,) array, trajsize=..., verbose=False)` |
| actions, angles, frequencies | `J, theta, Omega = agama.ActionFinder(pot)(xv, angles=True)`; columns $(r, z, \phi)$ |

## Exercises

1. **Mixing time.** Repeat §6.3 with patches of different sizes in $v_r$ (10, 40, 160 km/s). Measure the time at
   which $|\langle e^{i\theta_r}\rangle|$ first falls below 0.1 and compare it with $2\pi/\Delta\Omega_r$.
2. **No mixing, again.** Repeat §6.3 in the harmonic potential. What happens to the peak coarse-grained
   density and to $|\langle e^{i\theta_r}\rangle|$? (`ActionFinder` works for the harmonic potential too.)
3. **Area in a different slice.** Liouville's theorem is about the 6-d volume. Our 2-d check worked because
   all stars had the same $L$. Give the patch a spread of $L$ and recompute the $(r, v_r)$ area. Is it still
   conserved? Why not?
4. **Kick strength.** Redo §6.4 with kicks of 5 and 30 km/s. Does the clock still work? Plot the spiral
   as the *difference* between the kicked and unkicked histograms to see the weak case.
5. **Colour the spiral.** Colour the kicked stars in $(z, v_z)$ by their mean $v_R$ in each pixel
   (`scipy.stats.binned_statistic_2d`) instead of by number. Is the spiral visible?

## References

- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), chapter 4.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, chapter 5 "Equilibria of collisionless stellar systems":
  https://galaxiesbook.org/chapters/I-04.-Equilibria-of-Collisionless-Stellar-Systems.html
- T. Antoja et al. (2018), "A dynamically young and perturbed Milky Way disk", Nature 561, 360:
  https://doi.org/10.1038/s41586-018-0510-7 (arXiv:1804.10196)
- E. Vasiliev (2019), AGAMA, MNRAS 482, 1525: https://doi.org/10.1093/mnras/sty2672